### Neural Network

#### Parameter Settings

In [ ]:
### Setting all (hyper)parameters for model
input_size = 17
first_hidden_size = 10
second_hidden_size = None
output_size = 4

num_epochs = 30
batch_size = 32
learning_rate = 0.001

# Check for GPU availability and set processing unit accordingly
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


#### Data Loader


In [ ]:
# iterator that loads data in batches (more memory efficient), maybe not necessary for project size here but good to know
train_loader = DataLoader(dataset=train_dataset,
                          batch_size=batch_size,
                          shuffle=True)
val_loader = DataLoader(dataset=val_dataset,
                        batch_size=batch_size,
                        shuffle=False)
test_loader = DataLoader(dataset=test_dataset,
                         batch_size=batch_size,
                         shuffle=False)

In [ ]:
# # Consistency Check:
# examples = iter(train_loader)
# example_feature, example_target = next(examples)
# print(f"Feature:{example_feature}\nTarget:{example_target}")

#### Model, Loss and Optimizer Setup

In [ ]:
#Fully Connected Feed Forward
class NeuralNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(NeuralNet, self).__init__()
        self.l1 = nn.Linear(input_size, first_hidden_size)
        self.relu = nn.ReLU()
        self.l2 = nn.Linear(first_hidden_size, output_size)
    
    def forward(self, x):
        x = self.l1(x)
        x = self.relu(x)
        x = self.l2(x)
        # no activation function for last layer as that would restrict output intervall!
        return x
        
# Create model and push it to run on the device  
model = NeuralNet(input_size, first_hidden_size, output_size).to(device)
print("Number Parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))
     
# Loss (MSE) and optimizer (SGD)
criterion = nn.MSELoss(reduction="sum")
# criterion = nn.L1Loss(reduction="sum")
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

Number Parameters: 224


#### Training Loop

In [ ]:
# n_total_steps = len(train_loader)

train_losses = []
val_losses = []
train_losses2 = []
for epoch in range(num_epochs):
    # forward + backward pass for each epoch
    # losses=[]
    for i, (features, targets) in enumerate(train_loader):
        # Push data to device
        features = features.to(device)
        targets = targets.to(device)
        
        #Forward pass
        outputs = model(features)
        loss = criterion(outputs, targets)
        
        #Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # losses.append(loss*len(features))
    # train_losses2.append(sum(losses)/len(train_dataset))
    # print(train_losses2[epoch])
    
    # apply model to validation data in current epoch  
    with torch.no_grad():
        # training data losses (note: if too slow due to second forward pass, calculate as running average instead)
        losses = []
        for features, targets in train_loader:
            features = features.to(device)
            targets = targets.to(device)
            
            outputs = model(features)
            loss = criterion(outputs, targets)*len(features)
            losses.append(loss)
        train_losses.append(sum(losses)/len(train_dataset))
        # print(losses)
        
        # validation data losses
        losses = []
        for features, targets in val_loader:
            features = features.to(device)
            targets = targets.to(device)
            
            outputs = model(features)
            loss = criterion(outputs, targets)*len(features)
            # print(len(features))
            losses.append(loss)
        val_losses.append(sum(losses)/len(val_dataset))
        # print(losses)
    print(f"Epoch:{epoch+1:>{3}}/{num_epochs},  Training Loss: {train_losses[epoch]:.4f},   Validation Loss: {val_losses[epoch]:.4f}")

Epoch:  1/30,  Training Loss: 19.1641,   Validation Loss: 102.6942
Epoch:  2/30,  Training Loss: 17.9115,   Validation Loss: 100.4729
Epoch:  3/30,  Training Loss: 17.0830,   Validation Loss: 100.3193
Epoch:  4/30,  Training Loss: 16.3113,   Validation Loss: 99.1786
Epoch:  5/30,  Training Loss: 15.7084,   Validation Loss: 98.9167
Epoch:  6/30,  Training Loss: 14.7595,   Validation Loss: 98.6304
Epoch:  7/30,  Training Loss: 14.3306,   Validation Loss: 98.3815
Epoch:  8/30,  Training Loss: 13.7839,   Validation Loss: 96.7783
Epoch:  9/30,  Training Loss: 13.4980,   Validation Loss: 95.7397
Epoch: 10/30,  Training Loss: 12.9593,   Validation Loss: 96.1323
Epoch: 11/30,  Training Loss: 13.0283,   Validation Loss: 96.9231
Epoch: 12/30,  Training Loss: 12.5929,   Validation Loss: 95.3213
Epoch: 13/30,  Training Loss: 12.4730,   Validation Loss: 94.6713
Epoch: 14/30,  Training Loss: 12.5500,   Validation Loss: 95.9934
Epoch: 15/30,  Training Loss: 12.0361,   Validation Loss: 94.7975
Epoch: 